✅ Étape 3 : Visualisation des courbes + détection automatique des anomalies temporelles

Objectifs :

    📈 Tracer la courbe Aire segmentée vs Temps pour chaque image masquée.

    ⚠️ Marquer automatiquement les frames suspectes :

        Saut brutal entre deux frames : différence > seuil_saut

        Valeur trop faible : aire ≤ seuil_faible

        (optionnel) Plateau anormal : ≥5 frames avec faible variation (peut être ajouté plus tard)

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

# 📂 Dossier contenant les fichiers segmentés corrigés
data_dir = '/home/amenacer/Stage/Data/TollSome Segmentation /Segmentation/resultas/tollsome 2D+T/maskedimg/Tollsome_output1'

# 🔍 Fonction pour détecter les anomalies
def detect_anomalies(areas, seuil_saut=25, seuil_min=10, seuil_plateau=5):
    anomalies = []
    diff = np.abs(np.diff(areas))
    for i in range(1, len(areas) - 1):
        if diff[i-1] > seuil_saut:
            anomalies.append(i)
    for i in range(len(areas)):
        if areas[i] <= seuil_min:
            anomalies.append(i)
    for i in range(len(areas) - seuil_plateau):
        if np.all(np.abs(np.diff(areas[i:i+seuil_plateau])) < 3):
            anomalies.extend(list(range(i, i + seuil_plateau)))
    return np.unique(anomalies)

# 📊 Charger le CSV des descripteurs
summary_path = '/home/amenacer/Stage/Data/TollSome Segmentation /Segmentation/resultas/tollsome 2D+T/features_masked/Tollsome_output1/features_summary_groupe1.csv'
df_summary = pd.read_csv(summary_path)
display(df_summary)

# 📈 Tracer les courbes avec anomalies pour chaque fichier
for filename in df_summary['Fichier']:
    print(f"📂 Analyse de : {filename}")
    filepath = os.path.join(data_dir, filename)
    
    # Charger les données
    img = nib.load(filepath)
    data = img.get_fdata()
    areas = np.array([np.sum(data[:, :, t] > 0) for t in range(data.shape[-1])])
    
    # Détecter les anomalies
    anom = detect_anomalies(areas)
    anom = np.array(anom)  # ✅ nécessaire pour indexation

    # Tracer la courbe
    plt.figure(figsize=(8, 5))
    plt.plot(areas, marker='o', label="Aire segmentée")
    if len(anom) > 0:
        plt.scatter(anom, areas[anom], color='red', label="Anomalies détectées", zorder=5)
    plt.title(f"Aire segmentée vs Temps - {filename}")
    plt.xlabel("Temps (frame)")
    plt.ylabel("Aire segmentée (pixels)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()
